In [19]:
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import re
import os
import time
import logging
from urllib.parse import urlparse
 
import numpy  as np
import pandas as pd
import joblib
 
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split
 
# ─────────────────────────────────────────────────────────────────────────────
# LOGGING CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_DIR = "/kaggle/input/datasets/hassaanmustafavi/phishing-urls-dataset"
OUTPUT_DIR  = "/kaggle/working/preprocessed_output"
 
# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
 
# Characters whose raw counts are individual features
SPECIAL_CHARS = ['.', '-', '_', '/', '?', '=', '@', '&', '!', '#', '%', '+']
CHAR_NAMES     = [
    'dot', 'hyphen', 'underscore', 'slash',
    'question', 'equals', 'at', 'ampersand',
    'exclaim', 'hash', 'percent', 'plus',
]
# Security-sensitive substrings → binary presence flag
SENSITIVE_WORDS = [
    'login', 'verify', 'bank', 'secure', 'update', 'account',
    'signin', 'password', 'confirm', 'ebayisapi', 'webscr',
    'paypal', 'free', 'lucky', 'service', 'bonus', 'redirect',
]
 
# Well-known URL shortening services
SHORTENING_SERVICES = {
    'bit.ly', 'goo.gl', 'tinyurl.com', 't.co', 'ow.ly',
    'is.gd', 'buff.ly', 'adf.ly', 'bit.do', 'mcaf.ee',
    'shorte.st', 'go2l.ink', 'x.co', 'ity.im', 'q.gs',
    'po.st', 'bc.vc', 'twitthis.com', 'u.to', 'j.mp',
    'buzurl.com', 'cutt.us', 'u.bb', 'yourls.org', 'x.co',
    'prettylinkpro.com', 'scrnch.me', 'filoops.info', 'vzturl.com',
}
 
# Regex: bare IPv4 address anywhere in the URL
_IP_PATTERN = re.compile(
    r'(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}'
    r'(?:25[0-5]|2[0-4]\d|[01]?\d\d?)'
)
_SENSITIVE_RE = re.compile(
    '|'.join(re.escape(w) for w in SENSITIVE_WORDS)
)
# Output directory for persisted artifacts
 
print("✓ Imports and configuration complete.")
print(f"  Dataset dir : {DATASET_DIR}")
print(f"  Output dir  : {OUTPUT_DIR}")
 

✓ Imports and configuration complete.
  Dataset dir : /kaggle/input/datasets/hassaanmustafavi/phishing-urls-dataset
  Output dir  : /kaggle/working/preprocessed_output


In [21]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — DISCOVER & LOAD CSV                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def discover_csv(directory: str) -> str:
    """
    Walk the dataset directory and return the path of the first .csv file.
    Prints all files found so you can see exactly what Kaggle mounted.
    Raises FileNotFoundError if no CSV is found.
    """
    print(f"\nContents of  {directory}/")
    print("─" * 50)
    all_files = []
    for root, dirs, files in os.walk(directory):
        for f in files:
            fpath = os.path.join(root, f)
            size  = os.path.getsize(fpath) / (1024 ** 2)
            print(f"  {fpath}  ({size:.1f} MB)")
            all_files.append(fpath)
    print("─" * 50)
 
    csvs = [f for f in all_files if f.lower().endswith('.csv')]
    if not csvs:
        raise FileNotFoundError(
            f"No .csv file found in {directory}.\n"
            "Check the dataset slug or re-add it via  Add Data → Search Datasets."
        )
    chosen = csvs[0]
    if len(csvs) > 1:
        print(f"⚠ Multiple CSVs found — using: {chosen}")
        print("  Others:", csvs[1:])
    else:
        print(f"✓ CSV found: {chosen}")
    return chosen
 
 
def load_dataset(filepath: str, chunksize: int = 100_000) -> pd.DataFrame:
    """
    Load CSV efficiently.
 
    • Always reads only the 'url' and 'label' columns (ignores extras).
    • Uses dtype hints to skip pandas type-inference overhead.
    • Chunks if file > 200 MB to stay within Kaggle's RAM limit.
    """
    size_mb = os.path.getsize(filepath) / (1024 ** 2)
    print(f"\nLoading  {os.path.basename(filepath)}  ({size_mb:.1f} MB) …")
    t0 = time.perf_counter()
 
    # Peek at header to confirm column names
    header = pd.read_csv(filepath, nrows=0)
    print(f"  Raw columns found : {header.columns.tolist()}")
 
    # Normalise column lookup (case-insensitive)
    col_lower = {c.lower(): c for c in header.columns}
    url_col   = col_lower.get('url',   None)
    lbl_col   = col_lower.get('label', None)
 
    if url_col is None or lbl_col is None:
        # Fallback: assume first col = url, second col = label
        url_col = header.columns[0]
        lbl_col = header.columns[1]
        print(f"  ⚠ 'url'/'label' not found by name — using: '{url_col}', '{lbl_col}'")
    else:
        print(f"  ✓ Mapped columns   :  url='{url_col}'  |  label='{lbl_col}'")
 
    read_kwargs = dict(
        usecols   = [url_col, lbl_col],
        dtype     = {url_col: "string", lbl_col: "string"},
        engine    = "c",
        na_values = ["", "NA", "N/A", "null", "NULL", "none", "None"],
        low_memory= False,
    )
 
    if size_mb > 200:
        print(f"  Large file — reading in chunks of {chunksize:,} rows …")
        chunks = pd.read_csv(filepath, chunksize=chunksize, **read_kwargs)
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(filepath, **read_kwargs)
 
    # Standardise to lowercase column names
    df.columns = ['url', 'label']
 
    elapsed = time.perf_counter() - t0
    print(f"  Loaded {len(df):,} rows in {elapsed:.2f} s.")
    return df
 
 
# ── Execute ───────────────────────────────────────────────────────────────────
csv_path = discover_csv(DATASET_DIR)
df_raw   = load_dataset(csv_path)
print("\nFirst 5 rows:")
display(df_raw.head())
 


Contents of  /kaggle/input/datasets/hassaanmustafavi/phishing-urls-dataset/
──────────────────────────────────────────────────
  /kaggle/input/datasets/hassaanmustafavi/phishing-urls-dataset/url_dataset.csv  (30.8 MB)
──────────────────────────────────────────────────
✓ CSV found: /kaggle/input/datasets/hassaanmustafavi/phishing-urls-dataset/url_dataset.csv

Loading  url_dataset.csv  (30.8 MB) …
  Raw columns found : ['url', 'type']
  ⚠ 'url'/'label' not found by name — using: 'url', 'type'
  Loaded 450,176 rows in 0.68 s.

First 5 rows:


,url,label
0,https://www.google.com,legitimate
1,https://www.youtube.com,legitimate
2,https://www.facebook.com,legitimate
3,https://www.baidu.com,legitimate
4,https://www.wikipedia.org,legitimate


In [22]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DATA CLEANING                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the raw DataFrame.
 
    Steps
    -----
    1. Print unique label values before normalisation (diagnostic).
    2. Drop rows with null url or label.
    3. Strip whitespace from both columns.
    4. Drop exact URL duplicates (keep first occurrence).
    5. Normalise labels to binary int:
         1 → phishing / bad / malicious / spam
         0 → benign   / good / safe / legitimate
    6. Drop rows with unrecognised labels and report count.
    """
    print("\n── Data Cleaning ────────────────────────────────────────────")
    print(f"  Input rows       : {len(df):,}")
 
    # ── Diagnostic: raw label distribution ──────────────────────────────────
    print(f"\n  Unique raw labels: {df['label'].unique().tolist()}")
    print(f"  Label value counts:\n{df['label'].value_counts().to_string()}")
 
    # ── 1. Drop nulls ────────────────────────────────────────────────────────
    df = df.dropna(subset=['url', 'label'])
    print(f"\n  After null drop  : {len(df):,} rows")
 
    # ── 2. Strip + lowercase ─────────────────────────────────────────────────
    df['url']   = df['url'].str.strip()
    df['label'] = df['label'].str.strip().str.lower()
 
    # ── 3. Drop duplicate URLs ───────────────────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=['url'])
    print(f"  After dedup      : {len(df):,} rows  (removed {before - len(df):,})")
 
    # ── 4. Label normalisation ───────────────────────────────────────────────
    label_map = {
        '1': 1,  '0': 0,
        'bad': 1, 'good': 0,
        'phishing': 1,   'benign': 0,
        'malicious': 1,  'safe': 0,
        'spam': 1,       'legitimate': 0,
        'malware': 1,    'clean': 0,
    }
    df['label'] = df['label'].map(label_map)
 
    # ── 5. Drop unrecognised labels ──────────────────────────────────────────
    unknown = df['label'].isna()
    if unknown.any():
        print(f"  ⚠ Dropping {unknown.sum():,} rows with unrecognised labels.")
        df = df[~unknown]
 
    df['label'] = df['label'].astype(np.int8)
 
    print(f"\n  Final clean rows : {len(df):,}")
    print(f"  Benign  (0)      : {(df['label'] == 0).sum():,}  "
          f"({100*(df['label']==0).mean():.1f}%)")
    print(f"  Phishing(1)      : {(df['label'] == 1).sum():,}  "
          f"({100*(df['label']==1).mean():.1f}%)")
    print("────────────────────────────────────────────────────────────")
 
    return df.reset_index(drop=True)
 
 
df_clean = clean_dataset(df_raw)
 


── Data Cleaning ────────────────────────────────────────────
  Input rows       : 450,176

  Unique raw labels: ['legitimate', 'phishing']
  Label value counts:
label
legitimate    345738
phishing      104438

  After null drop  : 450,176 rows
  After dedup      : 450,176 rows  (removed 0)

  Final clean rows : 450,176
  Benign  (0)      : 345,738  (76.8%)
  Phishing(1)      : 104,438  (23.2%)
────────────────────────────────────────────────────────────


In [23]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — LEXICAL FEATURE EXTRACTION (20 features)                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
# def _safe_parse(url: str) -> urlparse:
#     """Prepend http:// if no scheme — ensures urlparse separates netloc from path."""
#     if not str(url).startswith(("http://", "https://", "ftp://")):
#         url = "http://" + str(url)
#     return urlparse(url)
def _safe_parse(url: str):
    """
    Prepend http:// if no scheme — ensures urlparse separates netloc from path.
    Gracefully catches ValueError for malformed/spoofed URLs with bad bracket placement.
    """
    url_str = str(url)
    if not url_str.startswith(("http://", "https://", "ftp://")):
        url_str = "http://" + url_str
    try:
        return urlparse(url_str)
    except ValueError:
        # Phishing URLs often have unescaped brackets in the host area.
        # If urlparse fails, return an empty ParseResult object as a fallback.
        return urlparse("")
 
 
def extract_features_bulk(urls: pd.Series) -> pd.DataFrame:
    """
    Extract 20 lexical features from all URLs using vectorised pandas ops.
 
    Design notes
    ────────────
    • pandas .str methods (str.count, str.len, str.contains) are backed by
      C-level loops and are significantly faster than Python-level apply()
      for simple pattern matching over large Series.
 
    • urlparse components (netloc, path, query, fragment, scheme) are extracted
      once via apply() and stored as intermediate Series.  All subsequent
      feature derivations reuse these Series rather than re-parsing.
 
    • apply() is used only where true per-row logic is unavoidable:
        – IP regex  (re.search with compiled pattern)
        – Shortener set membership  (O(1) hash lookup)
 
    Feature list  (column order preserved)
    ────────────
    Structural  (7)
      url_length, hostname_length, path_length, query_length,
      fragment_length, num_subdomains, path_depth
 
    Character counts  (12)
      count_dot, count_hyphen, count_underscore, count_slash,
      count_question, count_equals, count_at, count_ampersand,
      count_exclaim, count_hash, count_percent, count_plus
 
    Statistical  (3)
      digit_count, letter_count, digit_letter_ratio
 
    Security signals — binary 0/1  (5)  [total = 27... wait, 7+12+3+5 = 27]
      has_ip, has_sensitive_word, is_shortened,
      https_in_hostname, uses_https
 
    Wait — actually 7+12+3+5 = 27 features total.
    The docstring says 20 but this version extracts 27.
    More is better for the model — keeping all.
    """
    print(f"\n── Feature Extraction ───────────────────────────────────────")
    print(f"  Processing {len(urls):,} URLs …")
    t0 = time.perf_counter()
 
    url_s  = urls.astype(str)
    url_lc = url_s.str.lower()
 
    # ── Parse URL components once ────────────────────────────────────────────
    parsed_series = url_s.apply(_safe_parse)
    netloc_s = parsed_series.apply(lambda p: p.netloc).str.lower()
    path_s   = parsed_series.apply(lambda p: p.path)
    query_s  = parsed_series.apply(lambda p: p.query)
    frag_s   = parsed_series.apply(lambda p: p.fragment)
    scheme_s = parsed_series.apply(lambda p: p.scheme).str.lower()
 
    feat = pd.DataFrame(index=urls.index)
 
    # ── Structural features ──────────────────────────────────────────────────
    feat['url_length']      = url_s.str.len()
    feat['hostname_length'] = netloc_s.str.len()
    feat['path_length']     = path_s.str.len()
    feat['query_length']    = query_s.str.len()
    feat['fragment_length'] = frag_s.str.len()
 
    # num_subdomains = dots in hostname - 1  (example.com has 1 dot → 0 subdomains)
    feat['num_subdomains']  = (netloc_s.str.count(r'\.') - 1).clip(lower=0)
    feat['path_depth']      = path_s.str.count('/')
 
    # ── Character count features ─────────────────────────────────────────────
    # regex-escaped versions for str.count()
    escaped_chars = [
        r'\.', r'\-', r'_',  r'/',  r'\?', r'=',
        r'@',  r'&',  r'!',  r'#',  r'%',  r'\+',
    ]
    for pattern, name in zip(escaped_chars, CHAR_NAMES):
        feat[f'count_{name}'] = url_s.str.count(pattern)
 
    # ── Lexical statistics ───────────────────────────────────────────────────
    feat['digit_count']        = url_s.str.count(r'\d')
    feat['letter_count']       = url_s.str.count(r'[a-zA-Z]')
    feat['digit_letter_ratio'] = feat['digit_count'] / (feat['letter_count'] + 1)
 
    # ── Security binary signals ──────────────────────────────────────────────
 
    # 1. IP address literal in URL
    feat['has_ip'] = url_s.apply(
        lambda u: int(bool(_IP_PATTERN.search(u)))
    )
 
    # 2. Security-sensitive word anywhere in URL (case-insensitive)
    feat['has_sensitive_word'] = url_lc.apply(
        lambda u: int(bool(_SENSITIVE_RE.search(u)))
    )
 
    # 3. URL shortening service as host
    bare_host = netloc_s.str.split(':').str[0]           # strip port if present
    feat['is_shortened'] = bare_host.apply(
        lambda h: int(h in SHORTENING_SERVICES)
    )
 
    # 4. 'https' token appears inside the hostname  (deceptive subdomains)
    #    e.g.  http://https-paypal-secure.com/login
    feat['https_in_hostname'] = netloc_s.str.contains(
        'https', regex=False
    ).astype(int)
 
    # 5. Actual HTTPS scheme
    feat['uses_https'] = (scheme_s == 'https').astype(int)
 
    # ── Sanity check — no NaN in feature matrix ──────────────────────────────
    nan_counts = feat.isna().sum()
    if nan_counts.any():
        print(f"  ⚠ NaN detected in features — filling with 0:\n{nan_counts[nan_counts > 0]}")
        feat = feat.fillna(0)
 
    elapsed = time.perf_counter() - t0
    print(f"  Done in {elapsed:.2f} s.")
    print(f"  Feature matrix shape : {feat.shape}")
    print(f"  Total features       : {feat.shape[1]}")
    print(f"  Columns :\n    {feat.columns.tolist()}")
    print("────────────────────────────────────────────────────────────")
    return feat
 
 
feat_df       = extract_features_bulk(df_clean['url'])
feature_names = feat_df.columns.tolist()
 
# Peek at a few rows
print("\nSample feature rows:")
display(feat_df.head(3))
display(feat_df.describe().round(3))
 


── Feature Extraction ───────────────────────────────────────
  Processing 450,176 URLs …
  Done in 16.10 s.
  Feature matrix shape : (450176, 27)
  Total features       : 27
  Columns :
    ['url_length', 'hostname_length', 'path_length', 'query_length', 'fragment_length', 'num_subdomains', 'path_depth', 'count_dot', 'count_hyphen', 'count_underscore', 'count_slash', 'count_question', 'count_equals', 'count_at', 'count_ampersand', 'count_exclaim', 'count_hash', 'count_percent', 'count_plus', 'digit_count', 'letter_count', 'digit_letter_ratio', 'has_ip', 'has_sensitive_word', 'is_shortened', 'https_in_hostname', 'uses_https']
────────────────────────────────────────────────────────────

Sample feature rows:


,url_length,hostname_length,path_length,query_length,fragment_length,num_subdomains,path_depth,count_dot,count_hyphen,count_underscore,...,count_percent,count_plus,digit_count,letter_count,digit_letter_ratio,has_ip,has_sensitive_word,is_shortened,https_in_hostname,uses_https
0,22,14,0,0,0,1,0,2,0,0,...,0,0,0,17,0.0,0,0,0,0,1
1,23,15,0,0,0,1,0,2,0,0,...,0,0,0,18,0.0,0,0,0,0,1
2,24,16,0,0,0,1,0,2,0,0,...,0,0,0,19,0.0,0,0,0,0,1


,url_length,hostname_length,path_length,query_length,fragment_length,num_subdomains,path_depth,count_dot,count_hyphen,count_underscore,...,count_percent,count_plus,digit_count,letter_count,digit_letter_ratio,has_ip,has_sensitive_word,is_shortened,https_in_hostname,uses_https
count,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,...,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000,450176.000
mean,60.238,19.294,27.438,5.537,0.031,1.121,2.417,2.621,1.252,0.420,...,0.090,0.068,4.194,45.502,0.089,0.009,0.064,0.001,0.000,0.782
std,37.572,6.690,24.678,28.326,1.704,0.670,1.522,1.145,2.571,1.338,...,1.167,0.570,9.222,27.004,0.187,0.097,0.245,0.036,0.015,0.413
min,8.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,4.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,40.000,15.000,11.000,0.000,0.000,1.000,1.000,2.000,0.000,0.000,...,0.000,0.000,0.000,31.000,0.000,0.000,0.000,0.000,0.000,1.000
50%,52.000,18.000,22.000,0.000,0.000,1.000,2.000,2.000,0.000,0.000,...,0.000,0.000,1.000,40.000,0.018,0.000,0.000,0.000,0.000,1.000
75%,71.000,22.000,38.000,0.000,0.000,1.000,3.000,3.000,1.000,0.000,...,0.000,0.000,6.000,54.000,0.118,0.000,0.000,0.000,0.000,1.000
max,2314.000,240.000,1897.000,2245.000,253.000,19.000,25.000,32.000,42.000,200.000,...,134.000,50.000,631.000,1839.000,4.709,1.000,1.000,1.000,1.000,1.000


In [24]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — FEATURE CORRELATION HEATMAP  (optional but useful)           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def plot_correlation_heatmap(feat_df: pd.DataFrame, label: pd.Series) -> None:
    """
    Plot a Pearson-correlation heatmap of all features + label.
    Helps identify redundant features before model training.
    """
    combined = feat_df.copy()
    combined['label'] = label.values
 
    corr = combined.corr(numeric_only=True)
 
    fig, ax = plt.subplots(figsize=(18, 14))
    sns.heatmap(
        corr,
        annot      = True,
        fmt        = ".2f",
        cmap       = "RdYlGn",
        center     = 0,
        linewidths = 0.4,
        ax         = ax,
        annot_kws  = {"size": 7},
    )
    ax.set_title("Feature Correlation Matrix (incl. label)", fontsize=14, pad=12)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/correlation_heatmap.png", dpi=120)
    plt.show()
    print("✓ Heatmap saved to output dir.")
 
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
plot_correlation_heatmap(feat_df, df_clean['label'])
 
 
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — STRATIFIED SPLIT (80/20)                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def stratified_split(
    X: np.ndarray,
    y: np.ndarray,
    test_size:    float = 0.20,
    random_state: int   = 42,
) -> tuple:
    """
    Stratified 80/20 split.
 
    'stratify=y' guarantees that both train and test partitions
    have the same phishing/benign ratio as the full dataset.
    This is essential for imbalanced datasets — without it, you
    risk putting 90% of phishing samples in one split and getting
    misleadingly high or low model metrics.
 
    Returns X_train, X_test, y_train, y_test  (all numpy arrays)
    """
    print("\n── Stratified Train/Test Split ──────────────────────────────")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size    = test_size,
        random_state = random_state,
        stratify     = y,
    )
    for split_name, y_split in [("Train", y_train), ("Test ", y_test)]:
        total = len(y_split)
        ph    = y_split.sum()
        print(f"  {split_name} : {total:>7,} rows  |  "
              f"benign={total - ph:,} ({100*(total-ph)/total:.1f}%)  "
              f"phishing={ph:,} ({100*ph/total:.1f}%)")
    print("────────────────────────────────────────────────────────────")
    return X_train, X_test, y_train, y_test
 
 
X = feat_df.values.astype(np.float64)
y = df_clean['label'].values.astype(np.int8)
 
X_train_raw, X_test_raw, y_train, y_test = stratified_split(X, y)
 

✓ Heatmap saved to output dir.

── Stratified Train/Test Split ──────────────────────────────
  Train : 360,140 rows  |  benign=276,590 (76.8%)  phishing=83,550 (23.2%)
  Test  :  90,036 rows  |  benign=69,148 (76.8%)  phishing=20,888 (23.2%)
────────────────────────────────────────────────────────────


In [25]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — FEATURE SCALING  (fit on train only — no leakage)            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def scale_features(
    X_train: np.ndarray,
    X_test:  np.ndarray,
) -> tuple:
    """
    StandardScaler: zero-mean, unit-variance normalisation.
 
    CRITICAL: scaler.fit_transform() is called on X_train ONLY.
              scaler.transform()       is called on X_test.
    Fitting on the full dataset before splitting would let the scaler
    see test-set statistics during training — that is data leakage.
 
    The persisted scaler.pkl is what you apply to new unseen URLs
    at inference time so the feature scale matches training.
 
    Returns X_train_scaled, X_test_scaled, fitted_scaler
    """
    print("\n── Feature Scaling (StandardScaler) ─────────────────────────")
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
 
    print(f"  Train mean ≈ 0 : {np.allclose(X_train_sc.mean(axis=0), 0, atol=1e-5)}")
    print(f"  Train std  ≈ 1 : {np.allclose(X_train_sc.std(axis=0),  1, atol=1e-2)}")
    print(f"  Train range    : [{X_train_sc.min():.3f}, {X_train_sc.max():.3f}]")
    print(f"  Test  range    : [{X_test_sc.min():.3f},  {X_test_sc.max():.3f}]")
    print("────────────────────────────────────────────────────────────")
    return X_train_sc, X_test_sc, scaler
 
 
X_train, X_test, scaler = scale_features(X_train_raw, X_test_raw)
 
 
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — SAVE ARTIFACTS                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def save_artifacts(
    X_train       : np.ndarray,
    X_test        : np.ndarray,
    y_train       : np.ndarray,
    y_test        : np.ndarray,
    scaler        : StandardScaler,
    feature_names : list,
    output_dir    : str = OUTPUT_DIR,
) -> None:
    """
    Persist all preprocessed artifacts to /kaggle/working/.
 
    Files saved
    ───────────
    X_train.pkl        – scaled float64 feature matrix (train)
    X_test.pkl         – scaled float64 feature matrix (test)
    y_train.pkl        – int8 label array (train)
    y_test.pkl         – int8 label array (test)
    scaler.pkl         – fitted StandardScaler (apply to new URLs at inference)
    feature_names.pkl  – ordered list of column names (maps column index → name)
 
    compress=3 uses zlib — ~40-60% smaller files, negligible extra time.
    """
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n── Saving Artifacts → {output_dir}/ ──────────────────────────")
 
    artifacts = {
        'X_train'       : X_train,
        'X_test'        : X_test,
        'y_train'       : y_train,
        'y_test'        : y_test,
        'scaler'        : scaler,
        'feature_names' : feature_names,
    }
 
    for name, obj in artifacts.items():
        path = os.path.join(output_dir, f"{name}.pkl")
        joblib.dump(obj, path, compress=3)
        size_kb = os.path.getsize(path) / 1024
        dtype   = getattr(obj, 'dtype', type(obj).__name__)
        shape   = getattr(obj, 'shape', f"len={len(obj)}" if hasattr(obj, '__len__') else '—')
        print(f"  ✓  {name:<20}  shape={str(shape):<20}  dtype={str(dtype):<10}  {size_kb:.1f} KB")
 
    print("────────────────────────────────────────────────────────────")
    print(f"✓ All artifacts saved.  Reload with:\n")
    print(f"  import joblib")
    print(f"  X_train = joblib.load('{output_dir}/X_train.pkl')")
    print(f"  X_test  = joblib.load('{output_dir}/X_test.pkl')")
    print(f"  y_train = joblib.load('{output_dir}/y_train.pkl')")
    print(f"  y_test  = joblib.load('{output_dir}/y_test.pkl')")
    print(f"  scaler  = joblib.load('{output_dir}/scaler.pkl')")
    print(f"  feature_names = joblib.load('{output_dir}/feature_names.pkl')")
 
 
save_artifacts(X_train, X_test, y_train, y_test, scaler, feature_names)
 
 


── Feature Scaling (StandardScaler) ─────────────────────────
  Train mean ≈ 0 : True
  Train std  ≈ 1 : True
  Train range    : [-2.878, 594.662]
  Test  range    : [-2.878,  152.661]
────────────────────────────────────────────────────────────

── Saving Artifacts → /kaggle/working/preprocessed_output/ ──────────────────────────
  ✓  X_train               shape=(360140, 27)          dtype=float64     9903.9 KB
  ✓  X_test                shape=(90036, 27)           dtype=float64     2473.0 KB
  ✓  y_train               shape=(360140,)             dtype=int8        62.3 KB
  ✓  y_test                shape=(90036,)              dtype=int8        15.9 KB
  ✓  scaler                shape=—                     dtype=StandardScaler  1.1 KB
  ✓  feature_names         shape=len=27                dtype=list        0.2 KB
────────────────────────────────────────────────────────────
✓ All artifacts saved.  Reload with:

  import joblib
  X_train = joblib.load('/kaggle/working/preprocessed_outpu

In [26]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — PIPELINE SUMMARY                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
print("\n" + "═" * 60)
print("  PREPROCESSING PIPELINE — COMPLETE SUMMARY")
print("═" * 60)
print(f"  Raw rows loaded       : {len(df_raw):>10,}")
print(f"  Rows after cleaning   : {len(df_clean):>10,}")
print(f"  Features extracted    : {len(feature_names):>10}")
print(f"  Training samples      : {X_train.shape[0]:>10,}")
print(f"  Testing  samples      : {X_test.shape[0]:>10,}")
print(f"  X_train shape         : {str(X_train.shape):>20}")
print(f"  X_test  shape         : {str(X_test.shape):>20}")
print(f"  y_train phishing rate : {100*y_train.mean():>9.2f}%")
print(f"  y_test  phishing rate : {100*y_test.mean():>9.2f}%")
print(f"  Output directory      : {OUTPUT_DIR}")
print("═" * 60)


════════════════════════════════════════════════════════════
  PREPROCESSING PIPELINE — COMPLETE SUMMARY
════════════════════════════════════════════════════════════
  Raw rows loaded       :    450,176
  Rows after cleaning   :    450,176
  Features extracted    :         27
  Training samples      :    360,140
  Testing  samples      :     90,036
  X_train shape         :         (360140, 27)
  X_test  shape         :          (90036, 27)
  y_train phishing rate :     23.20%
  y_test  phishing rate :     23.20%
  Output directory      : /kaggle/working/preprocessed_output
════════════════════════════════════════════════════════════


In [27]:
import os
from pathlib import Path

# Check current working directory
print(f"Current Working Directory: {os.getcwd()}")

# Check if the folder exists
target_dir = "./preprocessed_output"  # or "/kaggle/working/preprocessed_output"
print(f"Checking target directory: {os.path.abspath(target_dir)}")

if os.path.exists(target_dir):
    print("Folder exists! Contents:")
    files = os.listdir(target_dir)
    if files:
        for f in files:
            print(f"  - {f}")
    else:
        print("  (The folder is empty!)")
else:
    print("Folder does NOT exist at this path.")

Current Working Directory: /kaggle/working
Checking target directory: /kaggle/working/preprocessed_output
Folder exists! Contents:
  - X_test.pkl
  - correlation_heatmap.png
  - y_test.pkl
  - X_train.pkl
  - feature_names.pkl
  - y_train.pkl
  - scaler.pkl


In [28]:
import sys
import logging
import warnings
from pathlib import Path
 
warnings.filterwarnings("ignore")  
import numpy  as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")                      # non-interactive backend — safe on Kaggle/CI
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")          # applies clean grid style to all matplotlib figures
import shap

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
)
from xgboost import XGBClassifier
 
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,   # PR-AUC proxy
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.calibration import CalibratedClassifierCV

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# 1 ▸ LOGGING
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("train")

In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# 2 ▸ CONFIGURATION — tweak these without touching the rest of the file
# ─────────────────────────────────────────────────────────────────────────────
KAGGLE_PATH = "/kaggle/working/preprocessed_output"
LOCAL_PATH  = "./preprocessed_output"

ARTIFACT_DIR = "/kaggle/working/preprocessed_output"   # output of preprocess_kaggle1.py
OUTPUT_DIR   = "outputs"                               # PNG charts go here
MODEL_DIR    = "saved_models"                          # joblib exports
 
RANDOM_STATE    = 42
N_JOBS          = -1           # use all CPU cores
SHAP_SAMPLE_N   = 1_500        # SHAP subset size — keeps computation < 2 min
TOP_N_FEATURES  = 20           # how many features to show in importance charts
CV_FOLDS        = 5            # StackingClassifier cross-validation folds
log.info(f"Using artifact directory → {os.path.abspath(ARTIFACT_DIR)}")
# ─────────────────────────────────────────────────────────────────────────────
# 3 ▸ DIRECTORY SETUP
# ─────────────────────────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
log.info(f"Output directory  → {OUTPUT_DIR}/")
log.info(f"Model  directory  → {MODEL_DIR}/")
 

21:59:05  [INFO]  Using artifact directory → /kaggle/working/preprocessed_output
21:59:05  [INFO]  Output directory  → outputs/
21:59:05  [INFO]  Model  directory  → saved_models/


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# 4 ▸ HELPER UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def _banner(text: str, width: int = 70, char: str = "═") -> None:
    """Print a prominent section header to stdout."""
    print("\n" + char * width)
    print(f"  {text}")
    print(char * width)
 
 
def sanitise(X: np.ndarray, name: str = "array") -> np.ndarray:
    """
    Replace NaN / ±Inf with finite values.
    NaN  → column mean  (computed ignoring NaN)
    +Inf → column max   (ignoring Inf)
    -Inf → column min   (ignoring Inf)
    Reports the total number of imputed cells.
    """
    total_nan = np.isnan(X).sum()
    total_inf = np.isinf(X).sum()
    if total_nan + total_inf == 0:
        log.info(f"  {name}: no NaN / Inf detected ✓")
        return X
 
    log.warning(f"  {name}: {total_nan} NaN + {total_inf} Inf — imputing …")
    X = X.copy().astype(np.float64)
 
    # Per-column statistics ignoring bad values
    col_means = np.nanmean(np.where(np.isinf(X), np.nan, X), axis=0)
    col_max   = np.nanmax( np.where(np.isneginf(X) | np.isnan(X), np.nan, X), axis=0)
    col_min   = np.nanmin( np.where(np.isposinf(X) | np.isnan(X), np.nan, X), axis=0)
 
    # Fallback: if a whole column is bad, use 0
    col_means = np.nan_to_num(col_means, nan=0.0)
    col_max   = np.nan_to_num(col_max,   nan=0.0)
    col_min   = np.nan_to_num(col_min,   nan=0.0)
 
    nan_mask  = np.isnan(X)
    pinf_mask = np.isposinf(X)
    ninf_mask = np.isneginf(X)
 
    X[nan_mask]  = np.take(col_means, np.where(nan_mask)[1])
    X[pinf_mask] = np.take(col_max,   np.where(pinf_mask)[1])
    X[ninf_mask] = np.take(col_min,   np.where(ninf_mask)[1])
 
    log.info(f"  {name}: imputation complete ✓")
    return X
 
 
def elapsed(t0: float) -> str:
    """Return human-readable elapsed time string."""
    s = time.perf_counter() - t0
    return f"{s:.1f}s" if s < 60 else f"{s/60:.1f}min"
 
 

In [33]:
import os


In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# 5 ▸ LOAD PREPROCESSED ARTIFACTS
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 1 — LOADING PREPROCESSED ARTIFACTS")
 
def load_artifacts(artifact_dir: str) -> dict:
    """Load all .pkl files produced by the preprocessing notebook."""
    required = ["X_train", "X_test", "y_train", "y_test", "feature_names"]
    data = {}
    for key in required:
        path = os.path.join(artifact_dir, f"{key}.pkl")
        if not os.path.exists(path):
            log.error(f"Missing artifact: {path}")
            sys.exit(1)
        data[key] = joblib.load(path)
        shape_info = getattr(data[key], "shape", f"len={len(data[key])}" if hasattr(data[key], "__len__") else "—")
        log.info(f"  Loaded {key:<20}  shape={shape_info}")
    return data
 
arts          = load_artifacts(ARTIFACT_DIR)
X_train_raw   = arts["X_train"]
X_test_raw    = arts["X_test"]
y_train       = arts["y_train"].astype(np.int8)
y_test        = arts["y_test"].astype(np.int8)
feature_names = arts["feature_names"]
 
# ── Sanitise ─────────────────────────────────────────────────────────────────
X_train = sanitise(X_train_raw, "X_train")
X_test  = sanitise(X_test_raw,  "X_test")
 
print(f"\n  Train  : {X_train.shape[0]:>10,} samples × {X_train.shape[1]} features")
print(f"  Test   : {X_test.shape[0]:>10,} samples × {X_test.shape[1]} features")
print(f"  Phishing rate — train: {100*y_train.mean():.2f}%   test: {100*y_test.mean():.2f}%")
print(f"  Features: {feature_names}")
 
 
# ─────────────────────────────────────────────────────────────────────────────
# 6 ▸ MODEL DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 2 — DEFINING MODELS")
 
lr = LogisticRegression(
    C            = 1.0,
    solver       = "lbfgs",
    max_iter     = 1_000,
    class_weight = "balanced",     # handles 23% minority class
    random_state = RANDOM_STATE,
    n_jobs       = N_JOBS,
)
 
rf = RandomForestClassifier(
    n_estimators = 300,
    max_depth    = None,
    min_samples_leaf = 2,
    class_weight = "balanced_subsample",
    random_state = RANDOM_STATE,
    n_jobs       = N_JOBS,
)
 
xgb = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = (y_train == 0).sum() / (y_train == 1).sum(),  # handles imbalance
    eval_metric       = "logloss",
    use_label_encoder = False,
    random_state      = RANDOM_STATE,
    n_jobs            = N_JOBS,
    tree_method       = "hist",    # fastest CPU mode; auto-uses GPU if available
)
 
# ── Ensemble 1: Soft Voting ───────────────────────────────────────────────────
voting = VotingClassifier(
    estimators = [("lr", lr), ("rf", rf), ("xgb", xgb)],
    voting     = "soft",
    n_jobs     = N_JOBS,
)
 
# ── Ensemble 2: Stacking ─────────────────────────────────────────────────────
stacking = StackingClassifier(
    estimators     = [("lr", lr), ("rf", rf), ("xgb", xgb)],
    final_estimator = LogisticRegression(
        C            = 1.0,
        solver       = "lbfgs",
        max_iter     = 1_000,
        class_weight = "balanced",
        random_state = RANDOM_STATE,
    ),
    cv             = CV_FOLDS,
    stack_method   = "predict_proba",
    n_jobs         = N_JOBS,
    passthrough    = False,
)
 
# Registry: name → (model, needs_unscaled_features_for_shap)
MODELS = {
    "Logistic Regression" : lr,
    "Random Forest"       : rf,
    "XGBoost"             : xgb,
    "Voting Classifier"   : voting,
    "Stacking Classifier" : stacking,
}
 
log.info("Model registry:")
for name in MODELS:
    log.info(f"  ✓  {name}")
 


══════════════════════════════════════════════════════════════════════
  STEP 1 — LOADING PREPROCESSED ARTIFACTS
══════════════════════════════════════════════════════════════════════


22:05:32  [INFO]    Loaded X_train               shape=(360140, 27)
22:05:32  [INFO]    Loaded X_test                shape=(90036, 27)
22:05:32  [INFO]    Loaded y_train               shape=(360140,)
22:05:32  [INFO]    Loaded y_test                shape=(90036,)
22:05:32  [INFO]    Loaded feature_names         shape=len=27
22:05:32  [INFO]    X_train: no NaN / Inf detected ✓
22:05:32  [INFO]    X_test: no NaN / Inf detected ✓
22:05:32  [INFO]  Model registry:
22:05:32  [INFO]    ✓  Logistic Regression
22:05:32  [INFO]    ✓  Random Forest
22:05:32  [INFO]    ✓  XGBoost
22:05:32  [INFO]    ✓  Voting Classifier
22:05:32  [INFO]    ✓  Stacking Classifier



  Train  :    360,140 samples × 27 features
  Test   :     90,036 samples × 27 features
  Phishing rate — train: 23.20%   test: 23.20%
  Features: ['url_length', 'hostname_length', 'path_length', 'query_length', 'fragment_length', 'num_subdomains', 'path_depth', 'count_dot', 'count_hyphen', 'count_underscore', 'count_slash', 'count_question', 'count_equals', 'count_at', 'count_ampersand', 'count_exclaim', 'count_hash', 'count_percent', 'count_plus', 'digit_count', 'letter_count', 'digit_letter_ratio', 'has_ip', 'has_sensitive_word', 'is_shortened', 'https_in_hostname', 'uses_https']

══════════════════════════════════════════════════════════════════════
  STEP 2 — DEFINING MODELS
══════════════════════════════════════════════════════════════════════


In [39]:
# ─────────────────────────────────────────────────────────────────────────────
# 7 ▸ TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 3 — TRAINING ALL MODELS")

trained_models = {}

for name, model in MODELS.items():
    log.info(f"Training  [{name}] …")
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    dur = elapsed(t0)
    log.info(f"  ✓  [{name}]  done in {dur}")
    trained_models[name] = model

print("\n  All models trained successfully.")

22:05:37  [INFO]  Training  [Logistic Regression] …



══════════════════════════════════════════════════════════════════════
  STEP 3 — TRAINING ALL MODELS
══════════════════════════════════════════════════════════════════════


22:05:42  [INFO]    ✓  [Logistic Regression]  done in 5.0s
22:05:42  [INFO]  Training  [Random Forest] …
22:06:53  [INFO]    ✓  [Random Forest]  done in 1.2min
22:06:53  [INFO]  Training  [XGBoost] …
22:06:58  [INFO]    ✓  [XGBoost]  done in 4.9s
22:06:58  [INFO]  Training  [Voting Classifier] …
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [22:07:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
22:08:18  [INFO]    ✓  [Voting Classifier]  done in 1.3min
22:08:18  [INFO]  Training  [Stacking Classifier] …
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [22:08:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[22:09:41] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[22:09:42


  All models trained successfully.


In [40]:
# ─────────────────────────────────────────────────────────────────────────────
# 8 ▸ EVALUATION UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_model(
    name  : str,
    model,
    X     : np.ndarray,
    y     : np.ndarray,
) -> dict:
    """
    Compute the full evaluation metric suite for one model.
 
    Returns
    -------
    dict with keys:
        f1, precision, recall, pr_auc, roc_auc, accuracy,
        y_pred, y_prob (probability of positive class)
    """
    y_pred = model.predict(X)
 
    # Probability of positive class for AUC metrics
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        raw    = model.decision_function(X)
        y_prob = 1 / (1 + np.exp(-raw))   # sigmoid squash
    else:
        y_prob = y_pred.astype(float)
 
    return {
        "f1"        : f1_score(y, y_pred, zero_division=0),
        "precision" : precision_score(y, y_pred, zero_division=0),
        "recall"    : recall_score(y, y_pred, zero_division=0),
        "pr_auc"    : average_precision_score(y, y_prob),
        "roc_auc"   : roc_auc_score(y, y_prob),
        "accuracy"  : accuracy_score(y, y_pred),
        "y_pred"    : y_pred,
        "y_prob"    : y_prob,
    }
 
 
def plot_confusion_matrix(
    name    : str,
    y_true  : np.ndarray,
    y_pred  : np.ndarray,
    save_dir: str,
) -> None:
    """Save a labelled confusion matrix PNG for one model."""
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix = cm,
        display_labels   = ["Legitimate (0)", "Phishing (1)"],
    )
 
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(
        ax           = ax,
        cmap         = "Blues",
        colorbar     = True,
        values_format= "d",
    )
    ax.set_title(f"Confusion Matrix — {name}", fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout()
 
    safe_name = name.lower().replace(" ", "_")
    path = os.path.join(save_dir, f"confusion_matrix_{safe_name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    log.info(f"  Saved → {path}")
 
 

In [41]:
# ─────────────────────────────────────────────────────────────────────────────
# 9 ▸ RUN EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 4 — EVALUATING ALL MODELS")
 
results = {}
for name, model in trained_models.items():
    log.info(f"Evaluating  [{name}] …")
    metrics = evaluate_model(name, model, X_test, y_test)
    results[name] = metrics
 
    # Print per-model classification report
    print(f"\n{'─'*60}")
    print(f"  Model : {name}")
    print(f"{'─'*60}")
    print(classification_report(
        y_test,
        metrics["y_pred"],
        target_names=["Legitimate", "Phishing"],
        digits=4,
    ))
    print(f"  ★ F1-Score   : {metrics['f1']:.4f}   ← PRIMARY METRIC")
    print(f"    Precision  : {metrics['precision']:.4f}")
    print(f"    Recall     : {metrics['recall']:.4f}")
    print(f"    PR-AUC     : {metrics['pr_auc']:.4f}")
    print(f"    ROC-AUC    : {metrics['roc_auc']:.4f}")
    print(f"    Accuracy   : {metrics['accuracy']:.4f}")
 
    # Confusion matrix PNG
    plot_confusion_matrix(name, y_test, metrics["y_pred"], OUTPUT_DIR)
 
 
# ─────────────────────────────────────────────────────────────────────────────
# 10 ▸ COMPARATIVE MARKDOWN TABLE
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 5 — COMPARATIVE RESULTS TABLE")
 
COLS = ["f1", "precision", "recall", "pr_auc", "roc_auc", "accuracy"]
COL_LABELS = {
    "f1"        : "★ F1-Score",
    "precision" : "Precision",
    "recall"    : "Recall",
    "pr_auc"    : "PR-AUC",
    "roc_auc"   : "ROC-AUC",
    "accuracy"  : "Accuracy",
}
 
# Build rows
rows = []
for name, m in results.items():
    rows.append({
        "Model"      : name,
        **{COL_LABELS[c]: f"{m[c]:.4f}" for c in COLS},
    })
 
df_results = pd.DataFrame(rows).set_index("Model")
 
# ── Markdown print ────────────────────────────────────────────────────────────
header_row  = "| Model | " + " | ".join(df_results.columns) + " |"
divider_row = "| " + " | ".join(["---"] * (len(df_results.columns) + 1)) + " |"
print("\n" + header_row)
print(divider_row)
for idx, row in df_results.iterrows():
    print("| " + idx + " | " + " | ".join(row.values) + " |")
 
# ── Identify best model by F1 ─────────────────────────────────────────────────
best_name = max(results, key=lambda n: results[n]["f1"])
best_f1   = results[best_name]["f1"]
print(f"\n  ✦ Best model by F1-Score: [{best_name}]  F1 = {best_f1:.4f}")
 
# ── Save table as CSV ─────────────────────────────────────────────────────────
csv_path = os.path.join(OUTPUT_DIR, "results_comparison.csv")
df_results.to_csv(csv_path)
log.info(f"  Results table saved → {csv_path}")
 
 

22:16:07  [INFO]  Evaluating  [Logistic Regression] …



══════════════════════════════════════════════════════════════════════
  STEP 4 — EVALUATING ALL MODELS
══════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Model : Logistic Regression
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

  Legitimate     0.9883    0.9943    0.9913     69148
    Phishing     0.9808    0.9611    0.9708     20888

    accuracy                         0.9866     90036
   macro avg     0.9846    0.9777    0.9811     90036
weighted avg     0.9866    0.9866    0.9866     90036

  ★ F1-Score   : 0.9708   ← PRIMARY METRIC
    Precision  : 0.9808
    Recall     : 0.9611
    PR-AUC     : 0.9895
    ROC-AUC    : 0.9932
    Accuracy   : 0.9866


22:16:08  [INFO]    Saved → outputs/confusion_matrix_logistic_regression.png
22:16:08  [INFO]  Evaluating  [Random Forest] …
22:16:10  [INFO]    Saved → outputs/confusion_matrix_random_forest.png
22:16:10  [INFO]  Evaluating  [XGBoost] …



────────────────────────────────────────────────────────────
  Model : Random Forest
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

  Legitimate     0.9943    0.9980    0.9961     69148
    Phishing     0.9934    0.9810    0.9871     20888

    accuracy                         0.9941     90036
   macro avg     0.9938    0.9895    0.9916     90036
weighted avg     0.9941    0.9941    0.9941     90036

  ★ F1-Score   : 0.9871   ← PRIMARY METRIC
    Precision  : 0.9934
    Recall     : 0.9810
    PR-AUC     : 0.9967
    ROC-AUC    : 0.9981
    Accuracy   : 0.9941


22:16:11  [INFO]    Saved → outputs/confusion_matrix_xgboost.png
22:16:11  [INFO]  Evaluating  [Voting Classifier] …


...

22:16:14  [INFO]    Saved → outputs/confusion_matrix_voting_classifier.png
22:16:14  [INFO]  Evaluating  [Stacking Classifier] …



────────────────────────────────────────────────────────────
  Model : Voting Classifier
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

  Legitimate     0.9940    0.9981    0.9960     69148
    Phishing     0.9937    0.9799    0.9868     20888

    accuracy                         0.9939     90036
   macro avg     0.9939    0.9890    0.9914     90036
weighted avg     0.9939    0.9939    0.9939     90036

  ★ F1-Score   : 0.9868   ← PRIMARY METRIC
    Precision  : 0.9937
    Recall     : 0.9799
    PR-AUC     : 0.9963
    ROC-AUC    : 0.9978
    Accuracy   : 0.9939


22:16:17  [INFO]    Saved → outputs/confusion_matrix_stacking_classifier.png



────────────────────────────────────────────────────────────
  Model : Stacking Classifier
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

  Legitimate     0.9955    0.9962    0.9959     69148
    Phishing     0.9874    0.9851    0.9863     20888

    accuracy                         0.9936     90036
   macro avg     0.9915    0.9907    0.9911     90036
weighted avg     0.9936    0.9936    0.9936     90036

  ★ F1-Score   : 0.9863   ← PRIMARY METRIC
    Precision  : 0.9874
    Recall     : 0.9851
    PR-AUC     : 0.9967
    ROC-AUC    : 0.9981
    Accuracy   : 0.9936

══════════════════════════════════════════════════════════════════════
  STEP 5 — COMPARATIVE RESULTS TABLE
══════════════════════════════════════════════════════════════════════

| Model | ★ F1-Score | Precision | Recall | PR-AUC | ROC-AUC | Accuracy |
| --- | --- | --- | --- | --- | --- | --- |
| Logistic Regression | 0.9708 | 0.9808 | 0.9611 | 0.9895 

22:16:17  [INFO]    Results table saved → outputs/results_comparison.csv


In [42]:
# ─────────────────────────────────────────────────────────────────────────────
# 11 ▸ FEATURE IMPORTANCE — RF vs XGBoost side-by-side
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 6 — FEATURE IMPORTANCE CHARTS (RF & XGBoost)")
 
def _importance_df(model, feature_names: list, top_n: int = TOP_N_FEATURES) -> pd.DataFrame:
    """Extract and sort feature importances from a tree-based model."""
    importances = model.feature_importances_
    df = pd.DataFrame({
        "feature"    : feature_names,
        "importance" : importances,
    }).sort_values("importance", ascending=False).head(top_n)
    return df
 
 
rf_imp  = _importance_df(trained_models["Random Forest"], feature_names)
xgb_imp = _importance_df(trained_models["XGBoost"],       feature_names)
 
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle(
    f"Top-{TOP_N_FEATURES} Feature Importances",
    fontsize=16, fontweight="bold", y=1.01,
)
 
PALETTE = {"RF": "#2196F3", "XGB": "#FF5722"}
 
# ── Random Forest ─────────────────────────────────────────────────────────────
ax_rf = axes[0]
bars = ax_rf.barh(
    rf_imp["feature"][::-1],
    rf_imp["importance"][::-1],
    color    = PALETTE["RF"],
    edgecolor= "white",
    linewidth= 0.5,
)
ax_rf.set_title("Random Forest", fontsize=13, fontweight="bold")
ax_rf.set_xlabel("Mean Decrease in Impurity", fontsize=11)
ax_rf.set_ylabel("Feature", fontsize=11)
ax_rf.tick_params(axis="y", labelsize=9)
ax_rf.xaxis.grid(True, linestyle="--", alpha=0.5)
ax_rf.set_axisbelow(True)
# Value labels on bars
for bar, val in zip(bars, rf_imp["importance"][::-1]):
    ax_rf.text(
        bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", ha="left", fontsize=7.5, color="#333333",
    )
 
# ── XGBoost ───────────────────────────────────────────────────────────────────
ax_xgb = axes[1]
bars2 = ax_xgb.barh(
    xgb_imp["feature"][::-1],
    xgb_imp["importance"][::-1],
    color    = PALETTE["XGB"],
    edgecolor= "white",
    linewidth= 0.5,
)
ax_xgb.set_title("XGBoost", fontsize=13, fontweight="bold")
ax_xgb.set_xlabel("Feature Importance (gain)", fontsize=11)
ax_xgb.set_ylabel("Feature", fontsize=11)
ax_xgb.tick_params(axis="y", labelsize=9)
ax_xgb.xaxis.grid(True, linestyle="--", alpha=0.5)
ax_xgb.set_axisbelow(True)
for bar, val in zip(bars2, xgb_imp["importance"][::-1]):
    ax_xgb.text(
        bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", ha="left", fontsize=7.5, color="#333333",
    )
 
plt.tight_layout()
fi_path = os.path.join(OUTPUT_DIR, "feature_importance_rf_vs_xgb.png")
fig.savefig(fi_path, dpi=150, bbox_inches="tight")
plt.close(fig)
log.info(f"  Saved → {fi_path}")
 
 
# ─────────────────────────────────────────────────────────────────────────────
# 12 ▸ SHAP BEESWARM SUMMARY PLOT (XGBoost)
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 7 — SHAP SUMMARY PLOT (XGBoost)")
 
log.info(f"  Sampling {SHAP_SAMPLE_N} test rows for SHAP explanation …")
rng     = np.random.default_rng(RANDOM_STATE)
idx     = rng.choice(X_test.shape[0], size=min(SHAP_SAMPLE_N, X_test.shape[0]), replace=False)
X_shap  = X_test[idx]
 
log.info("  Building TreeExplainer …")
t_shap  = time.perf_counter()
explainer   = shap.TreeExplainer(trained_models["XGBoost"])
shap_values = explainer.shap_values(X_shap)
 
# shap_values may be a list (binary) or 2-D array depending on shap version
if isinstance(shap_values, list):
    sv = shap_values[1]   # positive class
else:
    sv = shap_values
 
log.info(f"  SHAP done in {elapsed(t_shap)}")
 
# Beeswarm summary plot
fig_shap, ax_shap = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    sv,
    X_shap,
    feature_names = feature_names,
    max_display   = TOP_N_FEATURES,
    show          = False,
    plot_type     = "dot",    # beeswarm
)
plt.title(
    f"SHAP Beeswarm — XGBoost (n={SHAP_SAMPLE_N} samples)",
    fontsize=13, fontweight="bold", pad=14,
)
plt.tight_layout()
shap_path = os.path.join(OUTPUT_DIR, "shap_beeswarm_xgboost.png")
plt.savefig(shap_path, dpi=150, bbox_inches="tight")
plt.close()
log.info(f"  Saved → {shap_path}")
 
 
# ─────────────────────────────────────────────────────────────────────────────
# 13 ▸ COMBINED METRICS BAR CHART (visual comparison)
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 8 — COMBINED METRICS BAR CHART")
 
metric_cols = ["f1", "precision", "recall", "pr_auc", "roc_auc", "accuracy"]
model_names = list(results.keys())
n_metrics   = len(metric_cols)
n_models    = len(model_names)
 
# Build matrix: rows=models, cols=metrics
score_matrix = np.array([
    [results[m][c] for c in metric_cols]
    for m in model_names
])
 
x       = np.arange(n_metrics)
width   = 0.15
offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width
 
COLORS = ["#3F51B5", "#4CAF50", "#FF5722", "#9C27B0", "#009688"]
LABELS = ["F1 ★", "Precision", "Recall", "PR-AUC", "ROC-AUC", "Accuracy"]
 
fig, ax = plt.subplots(figsize=(16, 7))
for i, (name, offset) in enumerate(zip(model_names, offsets)):
    ax.bar(
        x + offset,
        score_matrix[i],
        width,
        label     = name,
        color     = COLORS[i],
        edgecolor = "white",
        linewidth = 0.5,
        alpha     = 0.88,
    )
 
ax.set_title(
    "Model Comparison — All Metrics  (★ = Primary)",
    fontsize=14, fontweight="bold", pad=14,
)
ax.set_xticks(x)
ax.set_xticklabels(LABELS, fontsize=11)
ax.set_ylabel("Score", fontsize=12)
ax.set_ylim(0.5, 1.02)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
plt.tight_layout()
metrics_path = os.path.join(OUTPUT_DIR, "model_metrics_comparison.png")
fig.savefig(metrics_path, dpi=150, bbox_inches="tight")
plt.close(fig)
log.info(f"  Saved → {metrics_path}")
 
 


══════════════════════════════════════════════════════════════════════
  STEP 6 — FEATURE IMPORTANCE CHARTS (RF & XGBoost)
══════════════════════════════════════════════════════════════════════


22:16:44  [INFO]    Saved → outputs/feature_importance_rf_vs_xgb.png
22:16:44  [INFO]    Sampling 1500 test rows for SHAP explanation …
22:16:44  [INFO]    Building TreeExplainer …



══════════════════════════════════════════════════════════════════════
  STEP 7 — SHAP SUMMARY PLOT (XGBoost)
══════════════════════════════════════════════════════════════════════


22:16:45  [INFO]    SHAP done in 1.6s
22:16:46  [INFO]    Saved → outputs/shap_beeswarm_xgboost.png



══════════════════════════════════════════════════════════════════════
  STEP 8 — COMBINED METRICS BAR CHART
══════════════════════════════════════════════════════════════════════


22:16:47  [INFO]    Saved → outputs/model_metrics_comparison.png


In [43]:
# ─────────────────────────────────────────────────────────────────────────────
# 14 ▸ MODEL EXPORT (joblib)
# ─────────────────────────────────────────────────────────────────────────────
_banner("STEP 9 — SAVING MODELS TO DISK")
 
# ── Save all individual models ────────────────────────────────────────────────
for name, model in trained_models.items():
    safe = name.lower().replace(" ", "_")
    path = os.path.join(MODEL_DIR, f"{safe}.pkl")
    joblib.dump(model, path, compress=3)
    kb = os.path.getsize(path) / 1024
    log.info(f"  ✓  {name:<25}  → {path}  ({kb:.1f} KB)")
 
# ── Save best model with a canonical 'best_model.pkl' alias ──────────────────
best_model   = trained_models[best_name]
best_path    = os.path.join(MODEL_DIR, "best_model.pkl")
joblib.dump(best_model, best_path, compress=3)
log.info(f"  ✦  Best model [{best_name}] also saved as → {best_path}")
 
# ── Save a deployment bundle: model + feature_names ──────────────────────────
#    The Flask app can load this single file and have everything it needs.
bundle = {
    "model"         : best_model,
    "feature_names" : feature_names,
    "model_name"    : best_name,
    "f1_score"      : best_f1,
}
bundle_path = os.path.join(MODEL_DIR, "deployment_bundle.pkl")
joblib.dump(bundle, bundle_path, compress=3)
log.info(f"  ✦  Deployment bundle saved → {bundle_path}")
 
# ── Load verification ─────────────────────────────────────────────────────────
loaded  = joblib.load(bundle_path)
assert loaded["model_name"] == best_name, "Bundle load verification failed!"
log.info("  Bundle load verification passed ✓")
 
 
# ─────────────────────────────────────────────────────────────────────────────
# 15 ▸ FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
_banner("PIPELINE COMPLETE — FINAL SUMMARY", char="═")
 
print(f"\n  Trained Models     : {len(trained_models)}")
print(f"  Evaluation Metric  : F1-Score (primary)  |  Precision, Recall, PR-AUC, ROC-AUC, Accuracy")
print(f"  Test samples       : {X_test.shape[0]:,}")
print(f"  SHAP subset        : {SHAP_SAMPLE_N:,} samples (XGBoost)")
print(f"\n  {'─'*55}")
print(f"  {'Model':<28} {'F1-Score':>10}  {'ROC-AUC':>10}")
print(f"  {'─'*55}")
for name, m in sorted(results.items(), key=lambda kv: kv[1]["f1"], reverse=True):
    marker = "  ← BEST" if name == best_name else ""
    print(f"  {name:<28} {m['f1']:>10.4f}  {m['roc_auc']:>10.4f}{marker}")
print(f"  {'─'*55}")
 
print(f"\n  Outputs saved to   : {os.path.abspath(OUTPUT_DIR)}/")
print(f"  Models saved to    : {os.path.abspath(MODEL_DIR)}/")
print(f"\n  Files generated:")
for f in sorted(Path(OUTPUT_DIR).glob("*.png")):
    print(f"    📊  {f.name}")
for f in sorted(Path(OUTPUT_DIR).glob("*.csv")):
    print(f"    📋  {f.name}")
for f in sorted(Path(MODEL_DIR).glob("*.pkl")):
    kb = os.path.getsize(f) / 1024
    print(f"    💾  {f.name:<40}  ({kb:.1f} KB)")
 
print(f"\n  Flask deployment snippet:")
print(f"  ─────────────────────────────────────────────────────")
print(f"  import joblib, numpy as np")
print(f"  bundle  = joblib.load('{bundle_path}')")
print(f"  model   = bundle['model']")
print(f"  names   = bundle['feature_names']")
print(f"  # features = np.array([[...]])  # shape (1, {X_train.shape[1]})")
print(f"  # pred  = model.predict(features)")
print(f"  # proba = model.predict_proba(features)[:, 1]")
print(f"  ─────────────────────────────────────────────────────")
print(f"\n  ✦  Best Model : [{best_name}]   F1 = {best_f1:.4f}\n")
 

22:17:00  [INFO]    ✓  Logistic Regression        → saved_models/logistic_regression.pkl  (0.8 KB)



══════════════════════════════════════════════════════════════════════
  STEP 9 — SAVING MODELS TO DISK
══════════════════════════════════════════════════════════════════════


22:17:03  [INFO]    ✓  Random Forest              → saved_models/random_forest.pkl  (41114.0 KB)
22:17:03  [INFO]    ✓  XGBoost                    → saved_models/xgboost.pkl  (326.3 KB)
22:17:08  [INFO]    ✓  Voting Classifier          → saved_models/voting_classifier.pkl  (82894.9 KB)
22:17:14  [INFO]    ✓  Stacking Classifier        → saved_models/stacking_classifier.pkl  (82894.9 KB)
22:17:16  [INFO]    ✦  Best model [Random Forest] also saved as → saved_models/best_model.pkl
22:17:19  [INFO]    ✦  Deployment bundle saved → saved_models/deployment_bundle.pkl
22:17:20  [INFO]    Bundle load verification passed ✓



══════════════════════════════════════════════════════════════════════
  PIPELINE COMPLETE — FINAL SUMMARY
══════════════════════════════════════════════════════════════════════

  Trained Models     : 5
  Evaluation Metric  : F1-Score (primary)  |  Precision, Recall, PR-AUC, ROC-AUC, Accuracy
  Test samples       : 90,036
  SHAP subset        : 1,500 samples (XGBoost)

  ───────────────────────────────────────────────────────
  Model                          F1-Score     ROC-AUC
  ───────────────────────────────────────────────────────
  Random Forest                    0.9871      0.9981  ← BEST
  Voting Classifier                0.9868      0.9978
  XGBoost                          0.9866      0.9980
  Stacking Classifier              0.9863      0.9981
  Logistic Regression              0.9708      0.9932
  ───────────────────────────────────────────────────────

  Outputs saved to   : /kaggle/working/outputs/
  Models saved to    : /kaggle/working/saved_models/

  Files generated

In [44]:
import os
import shutil
from IPython.display import FileLink, display

# ─────────────────────────────────────────────────────────────────────────────
# ▸ FULL WORKSPACE BUNDLER & DOWNLOADER
# ─────────────────────────────────────────────────────────────────────────────
print("═" * 65)
print("  PREPARING WORKSPACE DOWNLOAD BUNDLE")
print("═" * 65)

# 1. Define paths
working_dir = '/kaggle/working'
staging_dir = '/kaggle/working/phishing_project_final_archive'
zip_filename = 'phishing_url_detection_project'
zip_filepath = f'/kaggle/working/{zip_filename}'

# Clean up any old staging directories if they exist
if os.path.exists(staging_dir):
    shutil.rmtree(staging_dir)
os.makedirs(staging_dir, exist_ok=True)

# 2. Select target folders to copy based on your image
target_folders = ['outputs', 'preprocessed_output', 'saved_models']
folders_added = 0

for folder in target_folders:
    source = os.path.join(working_dir, folder)
    destination = os.path.join(staging_dir, folder)
    
    if os.path.exists(source):
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print(f"  📂 Added: {folder}/")
        folders_added += 1
    else:
        print(f"  ⚠️ Warning: {folder}/ folder not found, skipping.")

# 3. Create the ZIP file if folders were bundled
if folders_added > 0:
    print("\n  📦 Compressing archive into a single ZIP file...")
    shutil.make_archive(zip_filepath, 'zip', staging_dir)
    
    # Clean up temporary staging folder to keep workspace tidy
    shutil.rmtree(staging_dir)
    
    print("═" * 65)
    print("  🎉 BUNDLE READY! Click the link below to download to your laptop:")
    print("═" * 65 + "\n")
    
    # 4. Render direct clickable download link
    display(FileLink(f"{zip_filename}.zip"))
else:
    print("❌ Error: No matching folders found to archive.")
    shutil.rmtree(staging_dir)

═════════════════════════════════════════════════════════════════
  PREPARING WORKSPACE DOWNLOAD BUNDLE
═════════════════════════════════════════════════════════════════
  📂 Added: outputs/
  📂 Added: preprocessed_output/
  📂 Added: saved_models/

  📦 Compressing archive into a single ZIP file...
═════════════════════════════════════════════════════════════════
  🎉 BUNDLE READY! Click the link below to download to your laptop:
═════════════════════════════════════════════════════════════════



/kaggle/working/phishing_url_detection_project.zip